<a href="https://colab.research.google.com/github/bilgeerol/bim448-sentiment/blob/main/notebooks/03_bert_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# 0. KURULUM VE DEPO İSKELETİ OLUŞTURMA
from google.colab import userdata, drive
import os

drive.mount('/content/drive')

TOKEN = userdata.get('GITHUB_TOKEN')
REPO_OWNER = "bilgeerol"
REPO = "bim448-sentiment"
REPO_DIR = f"/content/{REPO}"

if not os.path.exists(REPO_DIR):
    !git clone https://{TOKEN}@github.com/{REPO_OWNER}/{REPO}.git {REPO_DIR}
%cd {REPO_DIR}

!git config user.name "Bilge Erol"
!git config user.email "bilgeerol@example.com"

# Klasör yapısını ve .gitkeep dosyalarını oluştur
!mkdir -p data/raw data/processed notebooks src predictions results/tables results/figures results/logs results/hata_ornekleri report
!touch data/processed/.gitkeep notebooks/.gitkeep src/.gitkeep predictions/.gitkeep results/tables/.gitkeep results/figures/.gitkeep results/logs/.gitkeep results/hata_ornekleri/.gitkeep report/.gitkeep

# .gitignore dosyasını güncelle
gitignore_ek = """
# Proje özel kuralları
data/raw/
models/
checkpoints/
*.safetensors
*.bin
*.pt
.env
"""
with open(".gitignore", "a") as f:
    f.write(gitignore_ek)

# requirements.txt oluştur
reqs = """pandas
scikit-learn
matplotlib
seaborn
vaderSentiment
transformers
datasets
accelerate
torch
sentencepiece
sacremoses
groq
"""
with open("requirements.txt", "w") as f:
    f.write(reqs)

# Geçici README.md oluştur
readme_icerik = """# BİM 448 – Duygu Analizi Projesi (`bim448-sentiment`)

İngilizce ve Türkçe ürün yorumları üzerinde VADER, BERT ve LLM yöntemlerinin karşılaştırmalı duygu analizi.

## Ekip Üyeleri
- **Zehra:** Veri Hazırlama, VADER, Değerlendirme & Koordinasyon
- **Bilge:** GitHub Ortamı, BERT İnce Ayarı & Karşılaştırma Grafikleri
- **Sude:** LLM (Groq API), Hata Analizi & Video

## Çalıştırma Sırası
`01_veri_hazirlama.ipynb` → `02_vader.ipynb` / `03_bert_finetune.ipynb` / `04_llm.ipynb` → `05_degerlendirme.ipynb` → `06_karsilastirma.ipynb`
"""
with open("README.md", "w") as f:
    f.write(readme_icerik)

# GitHub'a gönder
!git add .
!git commit -m "[Bilge] Depo iskeleti, klasörler, requirements ve gitignore eklendi"
!git push

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into '/content/bim448-sentiment'...
/content/bim448-sentiment
[main (root-commit) 1cfafee] [Bilge] Depo iskeleti, klasörler, requirements ve gitignore eklendi
 12 files changed, 32 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 README.md
 create mode 100644 data/processed/.gitkeep
 create mode 100644 notebooks/.gitkeep
 create mode 100644 predictions/.gitkeep
 create mode 100644 report/.gitkeep
 create mode 100644 requirements.txt
 create mode 100644 results/figures/.gitkeep
 create mode 100644 results/hata_ornekleri/.gitkeep
 create mode 100644 results/logs/.gitkeep
 create mode 100644 results/tables/.gitkeep
 create mode 100644 src/.gitkeep
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (9/9), 1.18 KiB | 1.18 MiB/

In [3]:
import torch
print("GPU Kullanılabilir mi?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Adı:", torch.cuda.get_device_name(0))

GPU Kullanılabilir mi?: True
GPU Adı: Tesla T4


In [4]:
# 03_bert_finetune.ipynb — Parametreler ve Model İskeleti
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import pandas as pd
import os

# Dil seçimi: İngilizce için "en", Türkçe için "tr" yazabilirsin
LANG = "en"

# Model isimleri (Ortak kurallara göre sabit)
MODEL_NAME = {
    "en": "FacebookAI/roberta-base",
    "tr": "dbmdz/bert-base-turkish-cased"
}[LANG]

# Hiperparametreler
MAX_LEN = 128
LEARNING_RATE = 2e-5
BATCH_SIZE = 32
EPOCHS = 3
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
SEED = 42

OUTPUT_DIR = f"/content/drive/MyDrive/bim448/models/bert_{LANG}"
DEBUG = True  # Hızlı test için True (300 örnek), tam eğitim için False yapmalısın

LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

print(f"Seçilen Dil: {LANG.upper()}")
print(f"Kullanılacak Model: {MODEL_NAME}")
print(f"Çıkış Dizini: {OUTPUT_DIR}")

Seçilen Dil: EN
Kullanılacak Model: FacebookAI/roberta-base
Çıkış Dizini: /content/drive/MyDrive/bim448/models/bert_en


In [5]:
# 03_bert_finetune.ipynb — Tokenizer ve Veri Hazırlama Adımı
from transformers import AutoTokenizer

print(f"Model için tokenizer yükleniyor: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )

print("Tokenizer başarıyla hazırlandı! Şimdi veri setlerini okuma ve model yükleme adımına geçebiliriz.")

Model için tokenizer yükleniyor: FacebookAI/roberta-base


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizer başarıyla hazırlandı! Şimdi veri setlerini okuma ve model yükleme adımına geçebiliriz.


In [6]:
# 03_bert_finetune.ipynb — Model Mimarisini Yükleme
from transformers import AutoModelForSequenceClassification

print(f"Model yükleniyor ({MODEL_NAME})...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

# Modelin toplam parametre sayısını görelim
toplam_parametre = model.num_parameters()
print(f"Model başarıyla yüklendi!")
print(f"Toplam Parametre Sayısı: {toplam_parametre:,}")

Model yükleniyor (FacebookAI/roberta-base)...


model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model başarıyla yüklendi!
Toplam Parametre Sayısı: 124,647,939
